### NCF Visualization

In [ ]:
# Import necessary dependencies
import glob
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar
from tqdm import tqdm

sys.path.append('..')
from src.plot import animate_ncf_section_mesh, animate_fk_filtered_ncf_section_mesh, animate_directional_fk_ncf_section_mesh, animate_fv, animate_fv_pick, plot_scatter_section, plot_interpolated_section, plot_pcolormesh_section
from src.utils import parse_ncf_stack_filename
from src.disp import dispersion_curve, extr_disp, process_and_save_subset, regularize_dispersion_data, export_inversion_inputs
from IPython.display import HTML

import matplotlib as mpl
mpl.rcParams["animation.html"] = "jshtml"
mpl.rcParams["animation.embed_limit"] = 100.0  # Increases the limit to 100 MB

#### 1. Read NCFs

In [ ]:
# Choose dataset
pattern_v1_30d = "../data/ncf_stacks/30d/20210928_cc_*_30d_v1.npy"

files = sorted(glob.glob(pattern_v1_30d))
print("Matched files:", len(files))
print("First 3:", files[:3])

In [ ]:
# 1. Load ONE file to infer axes 
# We only need one representative NCF to build lag_axis & distance_axis
ncf0_path = files[0]
ncf0 = np.load(ncf0_path)

# Parse metadata (date, vs, window, mode)
date0, vs0, window0, mode0 = parse_ncf_stack_filename(ncf0_path)
print("Example parsed:", date0, vs0, window0, mode0)

In [ ]:
# 2. Define parameters
dt = 0.004      # sampling rate in sec (or 250 Hz)
fs = 250        # sampling frequency (Hz)

max_lag = 2.0   # seconds
npts_lag = int(max_lag * fs)

# Cable configuration (used only to create distance_axis length n_rec)
first_chan = 399
last_chan = 748
dx = 8.16
channels = np.arange(first_chan, last_chan + 1)

n_rec, n_lags = ncf0.shape

# Lag axis spans from –max_lag ... +max_lag
lag_axis = np.linspace(-max_lag, +max_lag, n_lags)

distance_axis = (channels - channels[0]) * dx

#### 2. Animate across ALL VS files 

In [ ]:
# Note: Because we are using FuncAnimation, the actual reading of the files
# and rendering of the frames happens during the `.to_jshtml()` call below.
ani = animate_ncf_section_mesh(
    pattern_v1_30d,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    mode="all",      # "all" | "causal" | "acausal"
    pclip=99,           # robust per-frame clipping (recommended)
    gauge_length=8.16,
    range_m=500,
    clip_lim=False,
    interval_ms=150,
)

# Render to the notebook 
HTML(ani.to_jshtml())

#### 3. Animate across ALL f-k VS files 

In [ ]:
# 1. Define f-k filter parameters
vmin = 100
vmax = 2000 

# 2. Call the animation function
# This will run the first progress bar (Pre-scanning + f-k filtering)
ani_fk = animate_fk_filtered_ncf_section_mesh(
    pattern=pattern_v1_30d, 
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    dt=dt,
    dx=dx,
    vmin=vmin,
    vmax=vmax,
    fk_mode="extract",       # Keep the data inside the cone
    fk_smooth="gaussian",    # Prevent ringing
    fk_sigma=2.0,            # Standard softness
    mode="all",              # Or "causal" / "acausal"
    pclip=99,                # Calculates the 99th percentile across all files for scaling
    clip_lim=False,          # Set to True if you want it to pan with the virtual source
    range_m=500              # Pan window (if clip_lim=True)
)

# 3. Render and display the video
# This will trigger the second progress bar (Rendering JSHTML Video)
HTML(ani_fk.to_jshtml())

#### 4. Animate across directional f-k files

In [ ]:
# Render the S1 (One-directional) wavefield
vmin = 100
vmax = 2000 

ani_s1 = animate_directional_fk_ncf_section_mesh(
    pattern=pattern_v1_30d, 
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    dt=dt,
    dx=dx,
    vmin=vmin,
    vmax=vmax,
    target="s1",             # <-- CHANGE THIS TO "s1", "s2", "causal", or "acausal"
    fk_mode="extract",       
    pclip=99,                
    clip_lim=False,          
    range_m=500,
    view_side="both",       # <-- CHANGE THIS TO "right", "left", or "both"
    pos_offset=0.0          # <--Skip the first X meters next to the VS             
)

HTML(ani_s1.to_jshtml())

In [ ]:
# Render the S1 (One-directional) wavefield
vmin = 100
vmax = 2000 

ani_s1 = animate_directional_fk_ncf_section_mesh(
    pattern=pattern_v1_30d, 
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    dt=dt,
    dx=dx,
    vmin=vmin,
    vmax=vmax,
    target="s1",             # <-- CHANGE THIS TO "s1", "s2", "causal", or "acausal"
    fk_mode="extract",       
    pclip=99,                
    clip_lim=True,          
    range_m=500, 
    view_side="right",       # <-- CHANGE THIS TO "right", "left", or "both"
    pos_offset=20.0          # <--Skip the first X meters next to the VS
)

HTML(ani_s1.to_jshtml())

In [ ]:
# Render the S1 (One-directional) wavefield
vmin = 100
vmax = 2000 

ani_s1 = animate_directional_fk_ncf_section_mesh(
    pattern=pattern_v1_30d, 
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    dt=dt,
    dx=dx,
    vmin=vmin,
    vmax=vmax,
    target="s1",             # <-- CHANGE THIS TO "s1", "s2", "causal", or "acausal"
    fk_mode="extract",       
    pclip=99,                
    clip_lim=True,          
    range_m=500, 
    view_side="left",       # <-- CHANGE THIS TO "right", "left", or "both"
    pos_offset=20.0          # <--Skip the first X meters next to the VS
)

HTML(ani_s1.to_jshtml())

#### 5. Select Good VSs

In [ ]:
# Format: (VS_start, VS_end, target, side)
# This allows you to pick 'right' for one range and 'left' for another.
selections = [
    (20, 70, "s1", "right"),  
    (70, 120, "s1", "left")
]

# Set the processing parameters 
vmin = 100    # Minimum velocity for f-k filter
vmax = 2000   # Maximum velocity for f-k filter
pos_offset = 0.0
range_m = 500.0
out_dir = "../data/ncf_disp"
fv_out_dir = "../results/fv_panels"
picks_out_dir = "../results/picks"
os.makedirs(fv_out_dir, exist_ok=True)
os.makedirs(picks_out_dir, exist_ok=True)

In [ ]:
tasks = []

for (start, end, target, side) in selections:
    # Get files that fall in this VS range
    subset_files = [
        p for p in files 
        if start <= int(parse_ncf_stack_filename(p)[1]) <= end
    ]
    
    for p in subset_files:
        task = process_and_save_subset(
            p, lag_axis, distance_axis, dt, dx, vmin, vmax,
            target=target, side=side, 
            pos_offset=pos_offset, range_m=range_m, out_dir=out_dir
        )
        tasks.append(task)

print(f"Total Subsets to process: {len(tasks)}")

# Run it!
with ProgressBar():
    results = dask.compute(*tasks)

print(f"Finished. Saved {len(results)} files to {out_dir}")

#### 6. Construct Dispersion Images

In [ ]:
# Frequency-Velocity Panel Parameters
fv_kwargs = {
    "vmin": 100.0,
    "vmax": 1200.0,
    "dv": 2.0,
    "fmin": 0.5,
    "fmax": 10.0,
    "normalize": True,
    "device": torch.device("cpu")
}

In [ ]:
processed_files = sorted(glob.glob(f"{out_dir}/*.npy"))
fv_results = {}

for fpath in tqdm(processed_files, desc="Computing & Saving f-v Panels"):
    # Load the pre-processed NCF subset (already flipped if it was 'left')
    item = np.load(fpath, allow_pickle=True).item()
    
    # 2. Compute f-v panel (Phase-Shift Method)
    fv, f_axis, v_axis = dispersion_curve(
        data=item["data"], 
        offset=item["dist_rel"], 
        t=item["lag"], 
        **fv_kwargs
    )
    
    # 3. Prepare metadata and filename
    fname = os.path.basename(fpath)
    save_name = fname.replace(".npy", "_fv.npy")
    save_path = os.path.join(fv_out_dir, save_name)
    
    # 4. Save to Disk for later use
    # We move tensors to CPU/Numpy for storage compatibility
    fv_data = {
        "fv": fv.cpu().numpy() if hasattr(fv, 'cpu') else fv,
        "f_axis": f_axis.cpu().numpy() if hasattr(f_axis, 'cpu') else f_axis,
        "v_axis": v_axis.cpu().numpy() if hasattr(v_axis, 'cpu') else v_axis,
        "side": item.get("side", "unknown"),
        "vs_m": item.get("vs_m", None)
    }
    
    np.save(save_path, fv_data)
    
    # Also keep in memory for immediate use
    fv_results[fname] = fv_data

print(f"Successfully saved {len(fv_results)} panels to {fv_out_dir}")

In [ ]:
ani_inspect = animate_fv(
    processed_files=processed_files, # Your raw subset files
    calc_fv_func=dispersion_curve,   # Your FV calculation math
    fv_kwargs=fv_kwargs,
    cmap="viridis"
)
HTML(ani_inspect.to_jshtml())

#### 7. Picks

In [ ]:
# Load just one to pick
# sample = np.load(all_fv_files[0], allow_pickle=True).item()
fv_files = sorted(glob.glob(os.path.join(fv_out_dir, "*_fv.npy")))
fv_files

In [ ]:
# Define manual packing parameters
# High SNR, smaller step is safer
# Lower SNR, might need larger step (e.g., 5)
pick_configs = {
    # --- Right Side Group (S1) ---
    "20210928_cc_020_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 3.0, 4.0, 4.9], "vmax_set": [600, 400, 390, 400], "f_mask": (1.5, 5.5), "step": 2},
    "20210928_cc_030_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 4.0], "vmax_set": [450, 400], "f_mask": (1.5, 5.5), "step": 2},
    "20210928_cc_040_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [450, 360, 330, 320], "f_mask": (1.5, 6.2), "step": 2},
    "20210928_cc_050_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [450, 400, 390, 390], "f_mask": (1.5, 6.2), "step": 2},
    "20210928_cc_060_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [450, 400, 390, 390], "f_mask": (1.5, 6.0), "step": 2},
    "20210928_cc_070_30d_v1_s1_right_fv": {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [450, 400, 390, 390], "f_mask": (1.5, 6.0), "step": 2},

    # --- Left Side Group (S1) ---
    "20210928_cc_070_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [450, 400, 390, 390], "f_mask": (1.5, 6.5), "step": 2},
    "20210928_cc_080_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 5.0, 5.5], "vmax_set": [550, 400, 390, 390], "f_mask": (2.0, 5.5), "step": 2},
    "20210928_cc_090_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 4.5], "vmax_set": [450, 400, 390], "f_mask": (1.5, 5.5), "step": 2},
    "20210928_cc_100_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 4.5], "vmax_set": [450, 400, 390], "f_mask": (2.0, 6.0), "step": 2},
    "20210928_cc_110_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 4.5], "vmax_set": [450, 400, 390], "f_mask": (1.5, 8.0), "step": 2},
    "20210928_cc_120_30d_v1_s1_left_fv":  {"f_ref_set": [2.0, 4.0, 4.5], "vmax_set": [450, 400, 390], "f_mask": (1.5, 6.0), "step": 2},
}

In [ ]:
# Run picking 
fv_files = sorted(glob.glob(os.path.join(fv_out_dir, "*_fv.npy")))

for fpath in tqdm(fv_files, desc="Picking Dispersion Curves"):
    fname_full = os.path.basename(fpath)
    
    # Identify which config to use
    matched_config = None
    for key in pick_configs:
        if key in fname_full:
            matched_config = pick_configs[key]
            break
            
    if not matched_config:
        continue
        
    # Load the f-v panel and axes
    data = np.load(fpath, allow_pickle=True).item()
    
    # Get individual parameters
    f_ref = matched_config["f_ref_set"]
    v_max_ref = matched_config["vmax_set"]
    f_min, f_max = matched_config["f_mask"]
    individual_step = matched_config.get("step", 3) 
    
    # Velocity Mute
    vmin_threshold = matched_config.get("vmin", 250) # Fallback to 250 m/s
    fv_panel_cleaned = data["fv"].copy()
    v_axis = data["v_axis"]
    
    # Find which indices are below our threshold
    low_v_mask = v_axis < vmin_threshold
    
    # Zero out the amplitudes in the fv matrix below the vmin threshold.
    # We check the shape to ensure we mute the correct axis.
    if fv_panel_cleaned.shape[0] == len(v_axis):
        fv_panel_cleaned[low_v_mask, :] = 0  # v is the rows
    else:
        fv_panel_cleaned[:, low_v_mask] = 0  # v is the columns
    
    try:
        # Run ridge tracking algorithm on the CLEANED panel
        raw_v_picks = extr_disp(
            data["f_axis"], 
            data["v_axis"], 
            fv_panel_cleaned, # <--- Pass the muted panel here
            f_ref_set=f_ref, 
            vmax_set=v_max_ref, 
            step=individual_step
        )
        
        # Apply the frequency mask to keep only reliable data
        f_axis = data["f_axis"]
        mask = (f_axis >= f_min) & (f_axis <= f_max)
        
        # Build the result dictionary
        pick_result = {
            "f": f_axis[mask],
            "v": raw_v_picks[mask],
            "vs_m": data.get("vs_m"),
            "side": data.get("side"),
            "params": matched_config,
            "original_fv_file": fname_full
        }
        
        # Save to ../results/picks/
        out_name = fname_full.replace("_fv.npy", "_pick.npy")
        np.save(os.path.join(picks_out_dir, out_name), pick_result)
        
    except Exception as e:
        print(f"\n[Error] Failed to pick {fname_full}: {e}")

print(f"\nProcessing complete. Picks saved to: {os.path.abspath(picks_out_dir)}")

In [ ]:
# Make sure we are grabbing the pre-computed FV panels
fv_panels = sorted(glob.glob(os.path.join(fv_out_dir, "*_fv.npy")))

# Run the lightning-fast playback animator
ani_inspect = animate_fv_pick(
    fv_files=fv_panels,
    picks_dir=picks_out_dir,
    cmap="magma",       # I recommend 'magma' for phase velocity ridges!
    interval_ms=500     # Slower interval to let you inspect the dots
)

# Display it
HTML(ani_inspect.to_jshtml())

#### 7. Mapping Picks

In this step, we aggregate all of our individual 1D dispersion curves and map them into a 2D pseudo-section along the DAS array. This allows us to visualize how the subsurface velocity structure varies laterally across our survey line. 

By plotting the data with the **low frequencies at the bottom** and high frequencies at the top, we simulate a true geological depth section. Lower frequencies possess longer wavelengths that penetrate deeper into the earth, which typically correspond to higher phase velocities (stiffer bedrock). Conversely, high frequencies map the shallower, slower soil layers.

Looking at the raw scattered picks, we can observe **irregular sampling** across the array. Some physical positions lack data points at certain frequencies—or drop out entirely above/below certain thresholds—due to localized drops in Signal-to-Noise Ratio (SNR) or poor waveform fidelity. Because standard 1D depth inversion algorithms require clean, uniform inputs, we will address these gaps in the next step using interpolation to regularize our frequency axes.

In [ ]:
pick_files = sorted(glob.glob(os.path.join(picks_out_dir, "*_pick.npy")))

# List to hold our scatter plot data
x_dist = []
y_freq = []
z_vel = []
markers = [] # to distinguish S1 (right) and S2 (left)

for fpath in pick_files:
    fname = os.path.basename(fpath)

    # Extract the virtual shot index from the filename
    # Assuming standard format: '20210928_cc_020_30d_v1_s1_right_pick.npy'
    parts = fname.split('_')
    try:
        vs_idx = int(parts[2])  # '020' -> 20
    except ValueError:
        print(f"Skipping {fname}: could not parse VS index.")
        continue

    # Calculate the distance from the virtual source (VS)
    distance_m = vs_idx * dx

    # Load the picked data
    data = np.load(fpath, allow_pickle=True).item()
    f = data["f"]          # Frequency axis
    v = data["v"]          # Picked phase velocities

    # Determine side for marker styling
    is_s1 = "s1" in fname.lower()

    # Append to our master lists
    x_dist.extend([distance_m] * len(f))
    y_freq.extend(f)
    z_vel.extend(v)
    markers.extend(['o' if is_s1 else 's'] * len(f)) # 'o' for S1, square for S2

In [ ]:
plot_scatter_section(x_dist, y_freq, z_vel, title="2D Dispersion Pseudo-Section along DAS Array")

In [ ]:
plot_interpolated_section(x_dist, y_freq, z_vel, title='Interpolated 2D Dispersion Pseudo-Section', cmap='turbo', y_max=5.5, grid_res=200)
# plot_pcolormesh_section(x_dist, y_freq, z_vel, title="Interpolated 2D Pseudo-Section", y_max=5.5)

#### 8. Regularize Picks

Because 1D depth inversion algorithms (like MCMC or Neighborhood Algorithm) expect clean, uniform inputs, we cannot feed them our raw, irregularly sampled picks. We must address the gaps, missing points, and mismatched frequency axes across our virtual shots. 

To create a perfectly standardized dataset, we apply a regularization function that does the following:
1. **Averages Duplicates (S1 & S2):** If a single virtual shot has picks from both its left (S2) and right (S1) sides at the same frequency, we average their phase velocities. This combines the data and boosts our Signal-to-Noise Ratio (SNR).

2. **Mutes High Frequencies:** We drop any noisy, unreliable picks above our target limit (e.g., 6.0 Hz).

3. **Linear Interpolation:** We use SciPy's `interp1d` to map the remaining valid picks onto a perfectly spaced, standardized target frequency grid (e.g., 2.0 to 6.0 Hz with a step of 0.5 Hz).

4. **Edge Padding:** If a station is completely missing picks at the very top or bottom of our target range, the interpolator safely pads the gap by extending the nearest known velocity, preventing unphysical extrapolation spikes.

In [ ]:
x_reg, y_reg, z_reg, regularized_profiles = regularize_dispersion_data(x_dist, y_freq, z_vel, f_min=2.0, f_max=6.0, f_step=0.5)

In [ ]:
plot_scatter_section(x_reg, y_reg, z_reg, title="(Regularized) 2D Dispersion Pseudo-Section along DAS Array")

In [ ]:
plot_interpolated_section(x_reg, y_reg, z_reg, title='Regularized 2D Dispersion Pseudo-Section', cmap='turbo', y_max=5.5, grid_res=200)
# plot_pcolormesh_section(x_dist, y_freq, z_vel, title="Regularized 2D Pseudo-Section", y_max=5.5)

#### 9. Save Regularize Picks for Inversion

Now that our dispersion curves are smoothly regularized onto a uniform frequency axis, the final step in our surface-wave processing workflow is to export them for 1D depth inversion. 

Standard inversion algorithms (such as Geopsy's `dinver`, Computer Programs in Seismology/Surf96, or custom MCMC codes) calculate the 1D shear-wave velocity ($V_s$) profile one spatial location at a time. Because of this, they cannot accept a single bulk file containing the entire array's scattered data. 

Instead, inversion software expects an **individual text file for every single virtual shot position**. Each of these files must be strictly formatted with two columns: **Frequency (Hz)** and **Phase Velocity (m/s)**. 

In [ ]:
export_inversion_inputs(regularized_profiles, output_dir="../results/inv_inputs")